# Israeli legal eval on Kaggle (T4 x2): Gemma 3 27B on the 400 untested questions + all 10 cases

**`gemma3:27b-it-qat`** on the code with the fixes from Claude's review of the `gemma27b_case_fixes` run:

| Fix | Where | What the 29 Sept runs showed |
|---|---|---|
| BM25: each query word counts once, by its best-matching form | `legal/keyword.py` (`query_groups`), `legal/corpus_lexical.py` | a word starting with a prefix letter counted twice ("ביטול" = ביטול + יטול) and outranked rarer terms of art; the unit test failed on Kaggle in both runs |
| No context slots reserved for sections the plan names; they join the rerank pool and compete | `retrieve_planned`, `pick_hits` | the plan's guesses ("חוק הירושה ss.8, 12, 86", "חוק המכר ss.2-5") took 3 of 16 slots: case 04 went back to the Sale (Apartments) Law, case 05 lost ss.26, 27, 35 |
| The reranker scores against the question and the issues, not the plan's law names | `retrieve_planned` | a wrong law name lifted every excerpt that mentions it |
| A looked-up section must come from the named law itself when the corpus has it | `named_law_hits` | "חוק המכר" also matched "חוק המכר (דירות)" |
| The plan gives a law's year only when sure | `PLAN_PROMPT` | invented years (חוק עבודת נשים, 1952) |
| The yes/no check adds a missing label but never swaps one the model wrote | `check_label` | swaps fixed 2 labels and broke 3 (IL-212, IL-409 twice) |
| The work-file revision drops echoed prompt tags | `eval_cases.py` | case 03 ended with `</work_file>` |

**What runs** (`QUESTION_SET = "new400"`, `CASE_LIMIT = 10`, about **6 hours**):
- **all 10 cases**, first (about 55 min): 01-05 as in the earlier runs, 06-10 never tested before;
- **the 400 questions no earlier run answered** -- the whole eval minus the 29 Sept 100-question sample -- in a
  shuffled order, so that whatever part finishes is a fair sample of all nine categories (about 4 h 35 min at
  ~41 s a question).

`QUESTION_SET = "all500"` adds the 29 Sept 100 after the 400 (about 70 min more, ~6 h 40 min in all): the same
questions as before, so the fixes can be compared on them directly.

**Time guard**: answering stops `STOP_ANSWERING_AFTER_H` hours after this notebook starts (default 7.25), and
the reports and the results zip are still written -- so a run near the end of the GPU quota ends with results
instead of losing them. Every answer is saved as it is written; `RESUME_FROM` continues an interrupted run.

**No judging here**: send the results zip back to Claude for the judging. Step 6 prints the judge-free numbers
at once (multiple choice, yes/no labels, governing section retrieved, the cases' rubric sections retrieved,
deadline hedges, flagged values, timing).

## Before running
1. **Add-ons -> Secrets**: `GITHUB_TOKEN` with read access to `zananiri/AI-IZ` (the same secret as before), and
   `HF_TOKEN`, a Hugging Face read token (optional: the embedder and reranker download with it).
2. **Add Input**: the corpus dataset built by `notebooks/kaggle_legal_corpus_build.ipynb`
   (`legal_corpus_vectordb.zip` or its unzipped folder) -- the same one as the earlier runs.
3. Settings -> Accelerator: **GPU T4 x2** (the 27B model is split over both GPUs).
4. Settings -> Internet: **on**.
5. **Run All**, or **Save Version -> Save & Run All** so the run survives closing the browser; the results
   zip is then in the version's Output.

In [ ]:
import time
NOTEBOOK_STARTED = time.time()   # the time guard counts from here

BRANCH = "main"            # the branch to test (the fixes are on main)
MODELS = [
    {"tag": "gemma3:27b-it-qat", "thinking": False},
]
RUN_CASES = True        # the paralegal cases, answered first
CASE_LIMIT = 10         # all 10 cases: 01-05 as in the earlier runs, 06-10 never tested (no fix was tuned on them)
QUESTION_SET = "new400"  # "new400": the questions no earlier run answered; "all500": those, then the 29 Sept 100;
                        # "sample100": the 29 Sept 100 only; None: cases only
SHUFFLE_SEED = 2026     # question order: shuffled once, so any finished part is a fair sample
STOP_ANSWERING_AFTER_H = 7.25  # stop answering this many hours after the start (8 h quota - setup - reports - margin)
# The 29 Sept 100-question sample (corpus_gemma27b_fixes and, in part, gemma27b_case_fixes).
TESTED_IDS = ['IL-001', 'IL-011', 'IL-014', 'IL-061', 'IL-062', 'IL-113', 'IL-115', 'IL-153', 'IL-202', 'IL-205', 'IL-245', 'IL-247', 'IL-248', 'IL-253', 'IL-254', 'IL-256', 'IL-293', 'IL-294', 'IL-296', 'IL-297', 'IL-300', 'IL-353', 'IL-016', 'IL-024', 'IL-075', 'IL-126', 'IL-164', 'IL-168', 'IL-172', 'IL-212', 'IL-213', 'IL-214', 'IL-217', 'IL-219', 'IL-262', 'IL-306', 'IL-309', 'IL-368', 'IL-409', 'IL-410', 'IL-032', 'IL-087', 'IL-135', 'IL-136', 'IL-223', 'IL-267', 'IL-268', 'IL-269', 'IL-317', 'IL-321', 'IL-371', 'IL-415', 'IL-036', 'IL-040', 'IL-230', 'IL-274', 'IL-276', 'IL-325', 'IL-327', 'IL-420', 'IL-046', 'IL-049', 'IL-103', 'IL-105', 'IL-188', 'IL-193', 'IL-286', 'IL-331', 'IL-332', 'IL-339', 'IL-378', 'IL-380', 'IL-381', 'IL-382', 'IL-383', 'IL-426', 'IL-108', 'IL-110', 'IL-240', 'IL-388', 'IL-389', 'IL-391', 'IL-432', 'IL-060', 'IL-198', 'IL-347', 'IL-392', 'IL-452', 'IL-453', 'IL-455', 'IL-456', 'IL-461', 'IL-468', 'IL-470', 'IL-483', 'IL-484', 'IL-486', 'IL-487', 'IL-493', 'IL-496']
TOP_K = 12              # retrieved chunks per question (issue spotting: 16, set in eval_run.py)
CASE_TOP_K = 16         # retrieved chunks per case
CONTEXT_LENGTH = 16384  # tokens per call (Ollama num_ctx): prompt + answer
ANSWER_MAX_TOKENS = 6144
CASE_MAX_TOKENS = 6144
REQUEST_TIMEOUT_S = 1800  # per model call
RETRIEVAL_DEVICE = "cuda"  # embedder + reranker; falls back to the CPU by itself if the GPUs are full

RUN_NAME = "gemma27b_full"   # RUN_DIR (data/legal/eval_gemma27b/RUN_NAME) is set in step 1
RESUME_FROM = None      # a path (or "auto") to an earlier run's results zip/folder attached as Input; None = fresh

# The earlier 27B runs, judged by Claude (evals/*_judged/ on the PC), for step 6.
BASELINE = {
    "sample100": {"run": "corpus_gemma27b_fixes (29 Sept)", "judged_score_pct": 75.8, "multiple_choice": "14/16",
                  "yes_no_label": "15/18", "governing_section_retrieved_pct": 88.8, "plans_cut_off": 36,
                  "plans_with_section_numbers": 59},
    "cases": {
        "corpus_gemma27b_fixes": {"judged": {"case_01": 25, "case_02": 31, "case_03": 43, "case_04": 42, "case_05": 44},
                                  "rubric_sections_retrieved": {"case_01": "1/10", "case_02": "2/5", "case_03": "3/5",
                                                                "case_04": "1/9", "case_05": "4/9"},
                                  "deadline_hedges": {"case_01": 3, "case_02": 3, "case_03": 2, "case_04": 3, "case_05": 3}},
        "gemma27b_case_fixes": {"judged": {"case_01": 27, "case_02": 43, "case_03": 50, "case_04": 27, "case_05": 41},
                                "rubric_sections_retrieved": {"case_01": "2/10", "case_02": "2/5", "case_03": "4/5",
                                                              "case_04": "1/9", "case_05": "1/9"},
                                "deadline_hedges": {"case_01": 2, "case_02": 5, "case_03": 1, "case_04": 0, "case_05": 1}},
    },
}

## 1. Code, packages and the unit tests

In [ ]:
import os, pathlib, shutil, subprocess, json, time, urllib.request
from kaggle_secrets import UserSecretsClient

def sh(command, log=None, check=True, env=None):
    """Runs a shell command with its output streamed live (and appended to `log`); raises on
    failure when check=True -- unlike `!command`, which carries on silently."""
    proc = subprocess.Popen(["bash", "-o", "pipefail", "-c", command], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    out = open(log, "a", encoding="utf-8") if log else None
    for line in proc.stdout:
        print(line, end="", flush=True)
        if out:
            out.write(line)
    if out:
        out.close()
    if proc.wait() and check:
        raise RuntimeError(f"exit code {proc.returncode}: {command}")
    return proc.returncode

sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
assert len(gpus) >= 2, "Settings -> Accelerator: GPU T4 x2 -- the 27B model needs both GPUs"

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
# Hugging Face token (Add-ons -> Secrets -> HF_TOKEN, read access): the embedder and reranker download
# with it -- higher rate limits, no "unauthenticated requests" warning. Optional: without it they still download.
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
    print("HF_TOKEN: set")
except Exception:
    print("HF_TOKEN: not set (optional) -- Hugging Face downloads run unauthenticated")
REPO = "/kaggle/working/AI-IZ"
if not pathlib.Path(REPO, ".git").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    f"https://{token}@github.com/zananiri/AI-IZ.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", "https://github.com/zananiri/AI-IZ.git"], check=True)
os.chdir(REPO)
RUN_DIR = pathlib.Path(f"data/legal/eval_gemma27b/{RUN_NAME}")
RUN_DIR.mkdir(parents=True, exist_ok=True)
sh("git log --oneline -1")

sh('pip install -q uv')
sh(f'uv pip install -q --system -e ".[legal,legal-data]" pytest')

# The code this run depends on, tested first (a failure is printed, not fatal; the full suite has a few
# known failures unrelated to the Legal eval).
sh("python -m pytest -q -p no:cacheprovider tests/unit/test_legal_corpus_lexical.py "
   "tests/unit/test_legal_corpus_retrieval.py tests/unit/test_legal_eval_repairs.py tests/unit/test_llm_client.py",
   log=RUN_DIR / "unit_tests.log", check=False)

## 2. Ollama and the model

`OLLAMA_MAX_LOADED_MODELS=1`: one model at a time. Flash attention + an 8-bit KV cache halve the 16k context's
memory. `OLLAMA_KEEP_ALIVE=-1`: the model stays loaded between the scripts, so it isn't reloaded after the
retrieval models have taken their share of a GPU. The 27B (~18 GB) doesn't fit one T4 (15 GB), so Ollama
splits it over both.

In [ ]:
sh("apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null")
if not shutil.which("ollama"):
    sh("curl -fsSL https://ollama.com/install.sh | sh > /dev/null")

OLLAMA_LOG = RUN_DIR / "ollama.log"
if subprocess.run(["curl", "-sf", "http://localhost:11434/api/tags"], capture_output=True).returncode:
    ollama = subprocess.Popen(["ollama", "serve"], stdout=open(OLLAMA_LOG, "a"), stderr=subprocess.STDOUT,
                              env={**os.environ, "OLLAMA_MAX_LOADED_MODELS": "1", "OLLAMA_FLASH_ATTENTION": "1",
                                   "OLLAMA_KV_CACHE_TYPE": "q8_0", "OLLAMA_KEEP_ALIVE": "-1"})
    for _ in range(60):
        if not subprocess.run(["curl", "-sf", "http://localhost:11434/api/tags"], capture_output=True).returncode:
            break
        time.sleep(2)

for m in MODELS:
    sh(f"ollama pull {m['tag']}")
sh("ollama list")

## 3. The bulk legal corpus (attached input), its BM25 index, and the eval set (from the repo)

The corpus comes from the dataset attached in step 1, found wherever it sits under `/kaggle/input` (unpacked or
zipped); the run stops if it's missing or a collection is empty. BM25 reads each category's `lexical_<category>.jsonl`
(written by `vectorize.py --export-lexical`, part of the corpus build); if a corpus has only the state
database, it's exported here. The index is built once (a few minutes) and cached beside it. Also checks every
law and section the gold answers cite against the installed corpus (`corpus_coverage.md`).

In [ ]:
# The corpus database: category folders (laws/, procedural_rules/) that hold chroma.sqlite3, found
# anywhere under /kaggle/input -- unpacked (Kaggle unzips uploads) or still zipped, at any folder depth.
# Folders of .jsonl records (the build's legal_corpus_jsonl) have the same category names but no
# chroma.sqlite3: they are ignored. The 28 Sept run copied that folder instead and ran without a corpus.
import zipfile
INPUT = pathlib.Path("/kaggle/input")
VECTORDB = pathlib.Path("data/legal_corpus_vectordb")
CATEGORIES_WANTED = ("laws", "procedural_rules")
db_roots = sorted({p.parent.parent for p in INPUT.rglob("chroma.sqlite3") if p.parent.name in CATEGORIES_WANTED})
print("database folders found:", [str(r) for r in db_roots] or "none")
shutil.rmtree(VECTORDB, ignore_errors=True)
VECTORDB.mkdir(parents=True)
if db_roots:
    for item in db_roots[0].iterdir():
        (shutil.copytree if item.is_dir() else shutil.copy2)(item, VECTORDB / item.name)
    print("copied", db_roots[0], "->", VECTORDB)
else:
    for z in sorted(INPUT.rglob("*.zip")):
        with zipfile.ZipFile(z) as archive:
            dbs = [n for n in archive.namelist()
                   if n.endswith("chroma.sqlite3") and len(n.split("/")) >= 2 and n.split("/")[-2] in CATEGORIES_WANTED]
            if not dbs:
                continue
            parts = dbs[0].split("/")  # [..., <category>, "chroma.sqlite3"]
            prefix = "/".join(parts[:-2]) + "/" if len(parts) > 2 else ""
            for member in archive.infolist():
                name = member.filename
                if name.startswith(prefix) and not name.endswith("/") and ".." not in name:
                    dest = VECTORDB / name[len(prefix):]
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    with archive.open(member) as fsrc, open(dest, "wb") as fdst:
                        shutil.copyfileobj(fsrc, fdst)
            print("unpacked", z, "->", VECTORDB)
            break

CORPUS_CATEGORIES = sorted(c for c in CATEGORIES_WANTED if (VECTORDB / c / "chroma.sqlite3").exists())
assert CORPUS_CATEGORIES, ("NO CORPUS DATABASE FOUND under /kaggle/input -- stopping instead of answering without it. "
                           "Attach the dataset with legal_corpus_vectordb (its laws/ and procedural_rules/ folders "
                           "must contain chroma.sqlite3).")
missing = [c for c in CATEGORIES_WANTED if c not in CORPUS_CATEGORIES]
if missing:
    print(f"[warn] the corpus has no {missing}: questions on those will retrieve nothing from them")

# Every collection must open and hold the chunks the build recorded.
BUILD_INFO = json.loads((VECTORDB / "_build_info.json").read_text()) if (VECTORDB / "_build_info.json").exists() else {}
import chromadb
bad = []
for c in CORPUS_CATEGORIES:
    n = chromadb.PersistentClient(path=str(VECTORDB / c)).get_collection(f"israeli_law_corpus_{c}").count()
    expected = BUILD_INFO.get("categories", {}).get(c, {}).get("chunks")
    print(f"{c:18s} {n:>9,} chunks" + (f" (build: {expected:,})" if expected else ""))
    if n == 0 or (expected and n != expected):
        bad.append(c)
if BUILD_INFO.get("chromadb_version") and BUILD_INFO["chromadb_version"] != chromadb.__version__:
    print(f"[note] built with chromadb {BUILD_INFO['chromadb_version']}, this machine has {chromadb.__version__}")
assert not bad, f"empty or incomplete collections {bad}: re-upload or rebuild the corpus before a long run"
print("corpus OK:", CORPUS_CATEGORIES)

# BM25 (one of the fixes under test): export the lexical copy if the corpus lacks it, then build and
# cache the index now -- and stop if it's missing, so the run doesn't quietly test without it.
no_lexical = [c for c in CORPUS_CATEGORIES if not (VECTORDB / c / f"lexical_{c}.jsonl").exists()]
if no_lexical:
    sh(f'python scripts/legal_data/vectorize.py --vectordb {VECTORDB} --categories {",".join(no_lexical)} --export-lexical',
       check=False)
started = time.time()
sh(f"""python - <<'EOF'
from docslides.legal.corpus_lexical import lexical_index
for category in {CORPUS_CATEGORIES!r}:
    index = lexical_index("{VECTORDB}", category)
    assert index is not None, f"no BM25 index for {{category}}: lexical_{{category}}.jsonl is missing"
    print(category, "BM25:", f"{{len(index.ids):,}} chunks, {{len(index.vocabulary):,}} terms")
EOF""")
print(f"BM25 ready in {time.time() - started:.0f}s")

EVAL_DIR = pathlib.Path("legal_txt/Evals")
zipfile.ZipFile(EVAL_DIR / "israeli_legal_eval.zip").extractall(EVAL_DIR)
EVAL = EVAL_DIR / "israeli_legal_eval"
assert (EVAL / "questions.jsonl").exists(), f"eval set not found under {EVAL}"

sh(f'python scripts/legal_data/check_corpus_coverage.py --gold {EVAL / "gold.jsonl"} --categories {",".join(CORPUS_CATEGORIES)} '
   f'--out "{RUN_DIR / "corpus_coverage.md"}" --json-out "{RUN_DIR / "corpus_coverage.json"}"', check=False)

## 4. The questions

The whole 500-question set minus the 29 Sept 100-question sample (`TESTED_IDS`) -- 400 questions no run has
answered -- shuffled with `SHUFFLE_SEED`, so a run the time guard cuts short still covers every category in
proportion. `all500` puts the 29 Sept 100 after them.

In [ ]:
import random
from collections import Counter, defaultdict

def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

all_questions = load_jsonl(EVAL / "questions.jsonl")
assert len(all_questions) == 500, f"expected the 500-question set, found {len(all_questions)}"
tested = set(TESTED_IDS)
new = [q for q in all_questions if q["id"] not in tested]
old = [q for q in all_questions if q["id"] in tested]
random.Random(SHUFFLE_SEED).shuffle(new)
random.Random(SHUFFLE_SEED).shuffle(old)
sample = {"new400": new, "all500": new + old, "sample100": old}.get(QUESTION_SET, [])
QUESTIONS = RUN_DIR / "questions_sample.jsonl"
with open(QUESTIONS, "w", encoding="utf-8") as f:
    for q in sample:
        f.write(json.dumps(q, ensure_ascii=False) + "\n")
print(f"{len(sample)} questions ({QUESTION_SET}) ->", QUESTIONS)
print("by category:", dict(Counter(q["category"] for q in sample).most_common()))
print(f"estimate: ~{len(sample) * 41 / 3600:.1f} h of questions after ~0.5 h for the cases")

## 4b. Resume: bring in an earlier run's results (if `RESUME_FROM` is set)

Copies the earlier `answers_*` files into place before anything runs, minus answers that came back empty or
errored -- so those get redone instead of being skipped as "already done".

In [ ]:
def model_dir(tag):
    return RUN_DIR / tag.replace(":", "_").replace("/", "_")

def find_eval_dir(root):
    first = model_dir(MODELS[0]["tag"]).name
    hits = sorted(p for p in pathlib.Path(root).rglob(RUN_NAME) if p.is_dir() and (p / first).exists())
    return hits[0] if hits else None

if RESUME_FROM:
    src = RESUME_FROM
    if src == "auto":
        zips = sorted(pathlib.Path("/kaggle/input").rglob(f"*{RUN_NAME}*results*.zip"))
        assert len(zips) <= 1, f"several earlier results attached, remove all but one from Input: {zips}"
        src = zips[0] if zips else "/kaggle/input"
    src = pathlib.Path(src)
    if src.is_file() and src.suffix == ".zip":
        extract_to = pathlib.Path("/kaggle/working/resume_from")
        shutil.rmtree(extract_to, ignore_errors=True)
        zipfile.ZipFile(src).extractall(extract_to)
        src = extract_to
    prev = find_eval_dir(src)
    assert prev, f"no {RUN_NAME}/<model> folders found under {src}"
    for m in MODELS:
        prev_dir, out_dir = prev / model_dir(m["tag"]).name, model_dir(m["tag"])
        for rel in ("bulk500/answers_rag.jsonl", "cases/answers_rag.jsonl"):
            src_file = prev_dir / rel
            if src_file.exists():
                (out_dir / rel).parent.mkdir(parents=True, exist_ok=True)
                shutil.copy(src_file, out_dir / rel)
        prev_trace, out_trace = prev_dir / "llm_trace", out_dir / "llm_trace"
        if prev_trace.exists():
            out_trace.mkdir(parents=True, exist_ok=True)
            for f in prev_trace.glob("*.jsonl"):
                shutil.copy(f, out_trace / f.name)
    print("resumed from", prev)

## 5. Answer (resumable)

Each model gets its own config (Ollama, its tag, thinking off, the 16k context, retrieval on the GPU) and
its own output folder and trace. The model is loaded into Ollama **first**, at the same context size the
scripts ask for, so it gets the GPU memory it needs; the embedder and reranker then go on whichever GPU has
more room left (or the CPU, if neither has -- the log line `retrieval ready on ...` says which). If the
reranker can't load at all, the scripts stop at start-up rather than answer on embedding distance alone.
Cases first, then the questions, under the time guard. Each line of output is `id (category): Ns`.

In [ ]:
import yaml

base_cfg = yaml.safe_load(open("config/config.yaml", encoding="utf-8"))

def model_env(m):
    out = model_dir(m["tag"])
    for sub in ("bulk500", "cases", "llm_trace"):
        (out / sub).mkdir(parents=True, exist_ok=True)
    cfg = json.loads(json.dumps(base_cfg))
    orch = cfg["legal"]["orchestrator"]
    orch.update(backend="ollama", base_url="http://localhost:11434", model=m["tag"],
                max_model_len=CONTEXT_LENGTH, request_timeout_s=REQUEST_TIMEOUT_S, supports_thinking=m["thinking"])
    cfg["legal"].setdefault("retrieval", {})["device"] = RETRIEVAL_DEVICE
    cfg.setdefault("logging", {})["llm_trace_dir"] = str(out / "llm_trace")
    cfg_path = out / "config.yaml"
    cfg_path.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding="utf-8")
    return out, {**os.environ, "DOCSLIDES_CONFIG": str(cfg_path),
                 "DOCSLIDES_LEGAL_ORCHESTRATOR_BACKEND": "ollama",
                 "DOCSLIDES_LEGAL_ORCHESTRATOR_BASE_URL": "http://localhost:11434",
                 "DOCSLIDES_LEGAL_ORCHESTRATOR_MODEL": m["tag"],
                 "DOCSLIDES_LEGAL_ORCHESTRATOR_MAX_MODEL_LEN": str(CONTEXT_LENGTH),
                 "DOCSLIDES_LEGAL_ORCHESTRATOR_SUPPORTS_THINKING": str(m["thinking"]).lower(),
                 "DOCSLIDES_LLM_TRACE_DIR": str(out / "llm_trace")}

def load_model(tag):
    """Loads `tag` into Ollama at CONTEXT_LENGTH (the num_ctx the scripts send, so their first call doesn't
    reload it) and shows how it was placed."""
    body = json.dumps({"model": tag, "prompt": "", "keep_alive": -1, "options": {"num_ctx": CONTEXT_LENGTH}}).encode()
    started = time.time()
    urllib.request.urlopen(urllib.request.Request("http://localhost:11434/api/generate", data=body,
                                                  headers={"Content-Type": "application/json"}), timeout=1800).read()
    print(f"{tag} loaded in {time.time() - started:.0f}s")
    sh("ollama ps; nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv")

def assert_full_retrieval(log):
    """Stops the run if retrieval fell back during this model's answers (the reranker didn't load). The
    scripts themselves stop at start-up if it can't load at all (warm_up_retrieval)."""
    text = log.read_text(encoding="utf-8", errors="replace") if log.exists() else ""
    if "legal_reranker_unavailable" in text:
        raise RuntimeError(f"the reranker was unavailable during this run ({log}): retrieval used embedding "
                           "distance only, so these answers aren't comparable -- fix the cause and run again")
    if "legal_retrieval_moved_to_cpu" in text:
        print("[note] retrieval ran on the CPU (no room on the GPUs beside the model): same results, slower")

def drop_failed(path):
    """Resume: remove answers that errored or came back empty, so this run redoes them."""
    if not path.exists():
        return 0
    rows = load_jsonl(path)
    good = [r for r in rows if not r.get("error") and str(r.get("answer") or r.get("work_file") or "").strip()]
    if len(good) < len(rows):
        with open(path, "w", encoding="utf-8") as f:
            for r in good:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    return len(rows) - len(good)

for m in MODELS:
    out, env = model_env(m)
    log = out / "run.log"
    no_think = "" if m["thinking"] else " --no-thinking"
    load_model(m["tag"])
    if RUN_CASES:
        cases = out / "cases" / "answers_rag.jsonl"
        print(f"\n{'=' * 20} {m['tag']}: {CASE_LIMIT} cases {'=' * 20}  (redoing {drop_failed(cases)} failed)")
        sh(f'python scripts/legal_data/eval_cases.py answer --dir {EVAL} --top-k {CASE_TOP_K} '
           f'--max-tokens {CASE_MAX_TOKENS} --limit {CASE_LIMIT} --out "{cases}"{no_think}', log=log, env=env)
    if sample:
        answers = out / "bulk500" / "answers_rag.jsonl"
        print(f"\n{'=' * 20} {m['tag']}: {len(sample)} questions {'=' * 20}  (redoing {drop_failed(answers)} failed)")
        # The time guard: eval_run.py writes each answer as it goes, so stopping it (SIGINT) loses
        # at most the question in progress.
        left_s = int(STOP_ANSWERING_AFTER_H * 3600 - (time.time() - NOTEBOOK_STARTED))
        if left_s < 120:
            print(f"[time guard] {STOP_ANSWERING_AFTER_H} h reached before the questions: skipping them")
        else:
            print(f"[time guard] answering for at most {left_s / 3600:.2f} h")
            code = sh(f'timeout --signal=INT --kill-after=60 {left_s} python scripts/legal_data/eval_run.py answer '
                      f'--questions "{QUESTIONS}" --top-k {TOP_K} --max-tokens {ANSWER_MAX_TOKENS} --out "{answers}"{no_think}',
                      log=log, env=env, check=False)
            if code == 124:
                print(f"[time guard] stopped after {STOP_ANSWERING_AFTER_H} h: "
                      f"{len(load_jsonl(answers)) if answers.exists() else 0} of {len(sample)} questions answered")
            elif code:
                raise RuntimeError(f"eval_run.py failed with exit code {code} -- see {log}")
    assert_full_retrieval(log)
    subprocess.run(["ollama", "stop", m["tag"]], capture_output=True)  # free the GPUs for the next model

## 6. Reasoning logs and the judge-free numbers

- `reasoning_report.md` / `reasoning_report_with_prompts.md`: every call's output, stop reason, timing (and prompts).
- `judge_free_report.*`: score.py (unmodified) without a judge -- multiple choice and yes/no labels.
- `comparison.md` / `summary.json`: the new questions, the 29 Sept 100 against their earlier numbers (with
  `all500`), and the cases against both earlier 27B runs: rubric sections retrieved, which are still missing,
  deadline hedges, flagged values, repairs, timing.

In [ ]:
import re, statistics

NIQQUD = re.compile(r"[\u0591-\u05C7]")
YEAR = re.compile(r"(?:^|\s)ה?תש[א-ת]{1,3}(?=\s|$)|\d{4}")
SECTION = re.compile(r"\d{1,4}[א-ת]{0,3}")
PLAN_CUT = re.compile(r"(?:^|[\s,\-–])ה?ת[א-ת]{0,3}\s*$")   # a law name that stops inside a Hebrew year
HEDGE = "יש לבדוק את המועד"
TEMPLATE_TOKEN = re.compile(r"</?(?:start_of_turn|end_of_turn|work_file|case_file)>")

def law_key(name):
    name = NIQQUD.sub("", (name or "").replace("־", " ").split(",")[0])
    name = re.sub(r"[\"'״׳`()\[\]\-–—\s]+", " ", name)
    name = YEAR.sub(" ", name)
    return " ".join(w[1:] if w.startswith("ה") and len(w) > 3 else w for w in name.split())

def same_law(a, b):
    a, b = law_key(a), law_key(b)
    return bool(a) and bool(b) and (a == b or a in b or b in a)

def base_section(s):
    found = SECTION.search(str(s or ""))
    return found.group(0) if found else None

def section_retrieved(law, section, retrieved):
    return any(same_law(law, r.get("title") or "") and (section is None or base_section(r.get("section")) == section)
               for r in retrieved or [])

gold = {g["id"]: g for g in load_jsonl(EVAL / "gold.jsonl")}
LAW_CODES = {}
for g in gold.values():
    for c in g.get("citations", []):
        if c.get("law_code"):
            LAW_CODES.setdefault(c["law_code"], c["law"])
cases_gold = {c["case_id"]: c for c in json.load(open(EVAL / "cases_gold.json", encoding="utf-8"))["cases"]}
category = {q["id"]: q["category"] for q in all_questions}

def governing_section_retrieved(row):
    cites = [c for c in gold.get(row["id"], {}).get("citations", []) if c.get("law")]
    if not cites:
        return None
    return any(section_retrieved(c["law"], base_section(c.get("section")), row.get("retrieved")) for c in cites)

def rubric_sections(case_id):
    pairs = set()
    for issue in cases_gold[case_id]["legal_issues"]:
        sections = [s.strip() for s in str(issue.get("section") or "").split(";") if s.strip()]
        codes = [c.strip() for c in issue["law"].split(";")]
        groups = [[s] for s in sections] if len(codes) > 1 and len(codes) == len(sections) else [sections] * len(codes)
        for code, secs in zip(codes, groups):
            if code in LAW_CODES:
                pairs |= {(LAW_CODES[code], base_section(s)) for s in secs or [None]}
    return sorted(pairs, key=str)

def question_stats(rows, report_rows):
    by_id = {r["id"]: r for r in report_rows}
    mcq = [by_id[a["id"]]["score"] for a in rows if a["id"] in by_id and by_id[a["id"]]["category"] == "mcq_bar"]
    labels = [by_id[a["id"]]["label_correct"] for a in rows if a["id"] in by_id and "label_correct" in by_id[a["id"]]]
    hits = [h for h in (governing_section_retrieved(a) for a in rows) if h is not None]
    plans = [a.get("plan") or [] for a in rows]
    return {"answers": len(rows), "errors": sum(bool(a.get("error")) for a in rows),
            "empty": sum(not str(a.get("answer") or "").strip() for a in rows),
            "multiple_choice": f"{sum(mcq):g}/{len(mcq)}", "yes_no_label": f"{sum(labels):g}/{len(labels)}",
            "governing_section_retrieved_pct": round(100 * sum(hits) / len(hits), 1) if hits else None,
            "plans_cut_off": sum(any(PLAN_CUT.search(i.get("law") or "") for i in p) for p in plans),
            "plans_with_section_numbers": sum(any(i.get("sections") for i in p) for p in plans),
            "repairs": dict(Counter(r for a in rows for r in a.get("repairs", []))),
            "stray_template_tokens": sum(bool(TEMPLATE_TOKEN.search(a.get("answer") or "")) for a in rows)}

summary = {"run": RUN_NAME, "question_set": QUESTION_SET, "questions_planned": len(sample),
           "cases": CASE_LIMIT if RUN_CASES else 0, "models": {}}
lines = [f"# Kaggle eval run {RUN_NAME}", "",
         f"{len(sample)} questions planned ({QUESTION_SET}), {summary['cases']} cases; "
         f"notebook ran {(time.time() - NOTEBOOK_STARTED) / 3600:.2f} h", ""]
for m in MODELS:
    out = model_dir(m["tag"])
    traces = sorted((out / "llm_trace").glob("*.jsonl"))
    if traces:
        files = " ".join(f'"{p}"' for p in traces)
        sh(f'python scripts/llm_trace_report.py {files} > "{out / "reasoning_report.md"}"', check=False)
        sh(f'python scripts/llm_trace_report.py {files} --prompts > "{out / "reasoning_report_with_prompts.md"}"', check=False)
    calls = [json.loads(l) for p in traces for l in open(p, encoding="utf-8") if l.strip()]
    by_site = defaultdict(list)
    seconds_by_job = defaultdict(float)
    for c in calls:
        by_site[c.get("call_site")].append(c)
        seconds_by_job[c.get("job_id")] += c.get("seconds") or 0

    # Questions
    answers = out / "bulk500" / "answers_rag.jsonl"
    ans = load_jsonl(answers) if answers.exists() else []
    report_rows = []
    if ans:
        sh(f'python {EVAL / "score.py"} report --gold {EVAL / "gold.jsonl"} --answers "{answers}" '
           f'--json-out "{out / "judge_free_report.json"}" > "{out / "judge_free_report.txt"}"', check=False)
        if (out / "judge_free_report.json").exists():
            report_rows = json.load(open(out / "judge_free_report.json", encoding="utf-8"))["rows"]
    q_all = question_stats(ans, report_rows)
    q_new = question_stats([a for a in ans if a["id"] not in tested], report_rows)
    q_old = question_stats([a for a in ans if a["id"] in tested], report_rows)
    per_q = [seconds_by_job[a["id"]] for a in ans if seconds_by_job.get(a["id"])]
    by_cat = Counter(category[a["id"]] for a in ans)

    # Cases
    case_rows = load_jsonl(out / "cases" / "answers_rag.jsonl") if (out / "cases" / "answers_rag.jsonl").exists() else []
    per_case = {}
    for row in case_rows:
        pairs = rubric_sections(row["case_id"])
        found = [p for p in pairs if section_retrieved(p[0], p[1], row.get("retrieved"))]
        text = row.get("work_file") or ""
        per_case[row["case_id"]] = {
            "rubric_sections_retrieved": f"{len(found)}/{len(pairs)}",
            "missed_sections": [f"{law.split(',')[0]} s.{sec}" for law, sec in pairs if (law, sec) not in found],
            "deadline_hedges": text.count(HEDGE), "flagged_values": ", ".join(row.get("unsupported_values") or []),
            "repairs": row.get("repairs") or [], "stray_template_tokens": bool(TEMPLATE_TOKEN.search(text)),
            "minutes": round(seconds_by_job.get(row["case_id"], 0) / 60, 1), "error": row.get("error")}

    call_table = {site: {"n": len(cs), "stopped_at_length": sum(x.get("done_reason") == "length" for x in cs),
                         "errors": sum(bool(x.get("error")) for x in cs),
                         "median_s": round(statistics.median([x.get("seconds") or 0 for x in cs]), 1),
                         "max_prompt_tokens": max((x.get("prompt_tokens") or 0) for x in cs),
                         "max_completion_tokens": max((x.get("completion_tokens") or 0) for x in cs)}
                  for site, cs in sorted(by_site.items())}
    summary["models"][m["tag"]] = {"questions": {"all": q_all, "new": q_new, "sample100": q_old,
                                                 "by_category": dict(by_cat),
                                                 "mean_model_s_per_question": round(statistics.mean(per_q), 1) if per_q else None},
                                   "cases": per_case, "calls": call_table}

    lines += [f"## {m['tag']}", "", f"### Questions: {len(ans)} of {len(sample)} answered", "",
              f"mean model time per question: {summary['models'][m['tag']]['questions']['mean_model_s_per_question']} s; "
              f"by category: {dict(by_cat.most_common())}", "",
              "| | new questions | the 29 Sept 100: then | now |", "|---|---|---|---|"]
    base = BASELINE["sample100"]
    for key in ("multiple_choice", "yes_no_label", "governing_section_retrieved_pct", "plans_cut_off",
                "plans_with_section_numbers"):
        lines.append(f"| {key.replace('_', ' ')} | {q_new[key]} | {base.get(key, '')} | {q_old[key] if q_old['answers'] else '-'} |")
    lines += [f"| judged score (%) | after Claude's judging | {base['judged_score_pct']} | after Claude's judging |", "",
              f"errors {q_all['errors']}, empty {q_all['empty']}; repairs {q_all['repairs']}; "
              f"answers with a stray template token: {q_all['stray_template_tokens']}", ""]
    if per_case:
        a, b = BASELINE["cases"]["corpus_gemma27b_fixes"], BASELINE["cases"]["gemma27b_case_fixes"]
        lines += ["### Cases", "",
                  "| case | judged (run 1 / run 2) | rubric sections retrieved (run 1 / run 2 / now) | deadline hedges (run 1 / run 2 / now) | flagged values | repairs | minutes |",
                  "|---|---|---|---|---|---|---|"]
        for cid, v in per_case.items():
            lines.append(f"| {cid} | {a['judged'].get(cid, '')} / {b['judged'].get(cid, '')} | "
                         f"{a['rubric_sections_retrieved'].get(cid, '')} / {b['rubric_sections_retrieved'].get(cid, '')} / {v['rubric_sections_retrieved']} | "
                         f"{a['deadline_hedges'].get(cid, '')} / {b['deadline_hedges'].get(cid, '')} / {v['deadline_hedges']} | "
                         f"{v['flagged_values'] or '-'} | {', '.join(v['repairs']) or '-'} | {v['minutes']} |")
        found = sum(int(v["rubric_sections_retrieved"].split("/")[0]) for v in per_case.values())
        total = sum(int(v["rubric_sections_retrieved"].split("/")[1]) for v in per_case.values())
        lines += ["", f"rubric sections retrieved in total: run 1 11/38, run 2 10/38, now {found}/{total}", ""]
        lines += [f"- {cid} still missing: {'; '.join(v['missed_sections'])}" for cid, v in per_case.items() if v["missed_sections"]]
        lines.append("")
    lines += ["| call site | calls | stopped at length | errors | median s | max prompt tok | max output tok |",
              "|---|---|---|---|---|---|---|"]
    lines += [f"| {site} | {x['n']} | {x['stopped_at_length']} | {x['errors']} | {x['median_s']} | {x['max_prompt_tokens']} | {x['max_completion_tokens']} |"
              for site, x in call_table.items()] + [""]

(RUN_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1), encoding="utf-8")
(RUN_DIR / "comparison.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines))

## 7. Environment record and the results zip (for Output)

In [ ]:
env_lines = []
for label, cmd in [("git", "git log -1 --format='%H %s'"), ("gpu", "nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv"),
                   ("ollama", "ollama --version"), ("models", "ollama list"),
                   ("python", "python -c \"import sys, chromadb, sentence_transformers, torch, sklearn; print(sys.version); print('chromadb', chromadb.__version__, '| sentence-transformers', sentence_transformers.__version__, '| torch', torch.__version__, '| scikit-learn', sklearn.__version__)\"")]:
    res = subprocess.run(["bash", "-c", cmd], capture_output=True, text=True)
    env_lines += [f"## {label}", (res.stdout + res.stderr).strip(), ""]
env_lines += ["## settings", json.dumps({k: globals()[k] for k in ("BRANCH", "MODELS", "QUESTION_SET", "SHUFFLE_SEED", "STOP_ANSWERING_AFTER_H", "RUN_CASES", "CASE_LIMIT",
              "TOP_K", "CASE_TOP_K", "CONTEXT_LENGTH", "ANSWER_MAX_TOKENS", "CASE_MAX_TOKENS", "REQUEST_TIMEOUT_S",
              "RETRIEVAL_DEVICE")}, indent=1)]
(RUN_DIR / "environment.txt").write_text("\n".join(env_lines), encoding="utf-8")

DEST = f"/kaggle/working/{RUN_NAME}_results.zip"
sh(f'cd /kaggle/working && zip -qr {DEST} AI-IZ/{RUN_DIR}')
print("results zip:", DEST, f"({pathlib.Path(DEST).stat().st_size / 1e6:.1f} MB)")
print("Send this zip back for the judging and the comparison with the earlier 27B runs.")